# Condenser performance and thermal capacity assessment

This notebook reproduces all six roadmap deliverables using the supplied ASHRAE RP-1043 archive. Run cells in order from the project root. The model predicts recorded condenser approach in kelvin. The lab-report comparison remains separate.

The archive, requirements, and scripts must accompany the notebook. No synthetic fallback is provided. The saved report explains assumptions, sensor consistency, energy balance, and the limits of failure-threshold claims.


## 1 Install the tested environment
Run this once in a fresh Python 3.12 environment. Restart the kernel if an already imported NumPy or pandas version changes.

In [ ]:
%pip install -r requirements.txt


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
assert (ROOT / 'ASHRAE RP-1043.zip').is_file(), 'Change to the project root containing the source ZIP.'
from scripts.run_analysis import main
main(ROOT)
DATA = ROOT / 'output/data'
FIGURES = ROOT / 'output/figures'


## 2 Data source selection and quality checks
The pipeline selects the Steady States sheets of 13 workbooks. It preserves original values and source identifiers, converts units to SI, and records exclusions. Normal cf Test 23 has a saturation/outlet-temperature inconsistency. Agreement of calculated water duty with Cond Tons is a unit check, not an independent validation.

In [ ]:
display(pd.read_csv(DATA / 'source_inventory.csv'))
audit = json.loads((DATA / 'data_audit.json').read_text())
display(pd.Series(audit, name='Audit'))
display(pd.read_csv(DATA / 'excluded_records.csv')[['source_file', 'Test', 'TCI', 'TCO', 'TRC', 'exclusion_reason']])
d = pd.read_csv(DATA / 'condenser_analysis.csv')
display(d[['source_file', 'condition_id', 'flow_kg_s', 'water_in_C', 'water_out_C', 'heat_duty_kW', 'recorded_approach_K']].head())


## 3 Deliverable 1 Operating-data plots
Family comparisons pool different loads. Matched comparisons use nominal Test identifiers and preserve real differences in achieved setpoints.

In [ ]:
display(Image(filename=str(FIGURES / '01_operating_data.png')))
display(Image(filename=str(FIGURES / '06_matched_comparisons.png')))
display(pd.read_csv(DATA / 'matched_run_comparisons.csv'))


## 4 Deliverable 2 Thermal bound and minimum flow

`Qupper = mass_flow * cp * (Tsat - Tin)` is an ideal bound at each measured saturation temperature. `effectiveness = Qwater / Qupper` describes utilization of that bound. It does not equal chiller COP. A finite condenser requires additional conductance and margin. The data do not establish an exact failure threshold.

The 0.85 screen is an illustrative planning assumption, not an approved control threshold.


In [ ]:
display(Image(filename=str(FIGURES / '02_thermal_limit.png')))
display(d[['source_file','condition_id','water_duty_kW','q_upper_kW','effectiveness','ideal_min_flow_kg_s']].sort_values('effectiveness', ascending=False).head(10))
display(pd.read_csv(DATA / 'illustrative_flow_screen.csv').query('exceeds_assumed_screen'))


## 5 Model validation
Seven whole operating-condition groups form the test set. Five grouped folds within the remaining training groups compare the Random Forest with mean and Ridge baselines. Targets and target-derived outlet/saturation temperatures are excluded from predictors. The saved final model is refit after evaluation.

In [ ]:
display(pd.read_csv(DATA / 'model_metrics.csv'))
display(Image(filename=str(FIGURES / '04_model_validation.png')))
metadata = json.loads((ROOT / 'output/models/model_metadata.json').read_text())
display(metadata)


## 6 Deliverable 3 Factor ranking
Permutation importance measures predictive dependence, not causal effects. Standard deviations across permutations are not confidence intervals. Tube-blockage fraction is known experimental metadata.

In [ ]:
display(pd.read_csv(DATA / 'factor_ranking.csv'))
display(Image(filename=str(FIGURES / '03_factor_ranking.png')))


## 7 Deliverable 4 What-if scenarios
Scenarios use normal1 Test 14 as a reference and the all-data fitted model. Small changes should be interpreted against model error. Range and local-neighbor checks cannot prove causal validity or physical feasibility.

In [ ]:
display(pd.read_csv(DATA / 'what_if_predictions.csv'))
display(Image(filename=str(FIGURES / '05_what_if.png')))
from scripts.predict import predict
predict(flow=16.9, inlet=24, load=191.1, evap_out=6.74, blockage=0)


## 8 Deliverable 5 Recommendations and lab checks
Pump cube-law estimates apply only to pump power. Plant-level savings require compressor and pump energy together. The lab-report values reproduce the source report, including its assumed flow; the energy balance remains unresolved.

In [ ]:
display(pd.read_csv(DATA / 'recommendations.csv'))
display(pd.read_csv(DATA / 'lab_report_values_and_checks.csv'))


## 9 Verification and report outputs
The written report and presentation accompany the course submission. This repository reproduces the data analysis, figures and model. Run the verification below after the pipeline.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/check_results.py'], check=True)
print('Analysis outputs:', ROOT / 'output')


## Sources

- Supplied ASHRAE RP-1043 archive, Steady States sheets and read me first.xls.
- Supplied HMT lab report, assignment, project roadmap and notebook photographs.
- Jin et al. (2019), RP-1043 variable definitions: https://arxiv.org/abs/1902.06366
- ASHRAE Handbook Condensers: https://handbook.ashrae.org/Handbooks/S20/IP/S20_Ch39/S20_ch39_ip.aspx
- ASHRAE Handbook Centrifugal Pumps: https://handbook.ashrae.org/Handbooks/S20/SI/S20_Ch44/S20_ch44_si.aspx
